# 6 · Taller 5 — Prototipo de Agente IA para la Universidad
### Plantilla de trabajo

**Complejidad: 🟢 Básica para empezar (puedes subir de complejidad según tu reto)**
**Dependencias base: solo `anthropic`**

Esta plantilla usa el patrón más simple y confiable (el loop manual del notebook 2) para que **ningún equipo se bloquee por instalación** de un framework. Si tu reto necesita RAG o MCP, agrega esas dependencias solo si las necesitas — mira los notebooks 4 y 5 para copiar el bloque de instalación correspondiente.

| Si tu reto necesita... | Copia las dependencias de... |
|---|---|
| Solo herramientas/acciones simples | Este notebook (`anthropic` únicamente) |
| Consultar documentos propios (RAG) | `llamaindex_rag.ipynb` |
| Exponer herramientas de forma estandarizada | `mcp_servidor_cliente.ipynb` |
| Memoria conversacional, multi-agente | `langchain_agente.ipynb` |


## 🎯 Objetivo de aprendizaje

Al terminar este notebook (y el taller) tu equipo va a poder:
- Definir con claridad el problema, el usuario y el criterio de éxito de un agente de IA para un caso real de la universidad.
- Diseñar e implementar las herramientas necesarias para ese agente, siguiendo el mismo patrón visto en los notebooks anteriores.
- Ejecutar casos de prueba y reflexionar críticamente sobre los riesgos y el nivel de autonomía apropiado para su prototipo.


## 📚 Teoría: de la charla al prototipo

Este taller integra todo lo visto en la sesión: el ciclo de un agente (notebook 2), la posibilidad de usar un framework si lo necesitas (notebook 3), RAG si tu reto requiere consultar documentos propios (notebook 4), y MCP si quieres exponer tus herramientas de forma estandarizada (notebook 5).

El proceso de diseño de un agente sigue siempre el mismo orden:
1. **Definir el caso de uso**: problema real, usuario, criterio de éxito.
2. **Diseñar las herramientas**: qué acciones o consultas necesita el agente, con una `description` clara para cada una.
3. **Implementar el loop**: razonar → actuar → observar, hasta una respuesta final.
4. **Probar**: casos de prueba concretos, no solo "que funcione una vez".
5. **Reflexionar**: qué riesgos tiene (loops infinitos, alucinación de herramientas, costos) y qué nivel de autonomía es el apropiado para este caso.

Este notebook usa deliberadamente el patrón más simple (el loop manual, sin framework) para que ningún equipo se bloquee por un problema de instalación durante el taller — agreguen complejidad (RAG, MCP, LangChain) solo si su reto realmente lo necesita.


## 0. Instalación (base mínima)

In [ ]:
!pip install -q anthropic

### Configurar API key de Anthropic

**Cómo obtenerla:** [console.anthropic.com](https://console.anthropic.com/settings/keys)

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `ANTHROPIC_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass("Pega tu ANTHROPIC_API_KEY: ")

print("API key configurada:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "FALTA")


## 1. El loop del agente (reutilizado del notebook 2)

In [ ]:
import anthropic

client = anthropic.Anthropic()

def run_agent(user_message: str, tools, implementations, max_steps: int = 5, verbose: bool = True):
    """Implementación manual del ciclo ReAct: razonar -> actuar -> observar."""
    messages = [{"role": "user", "content": user_message}]

    for step in range(1, max_steps + 1):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason != "tool_use":
            texto_final = "".join(b.text for b in response.content if b.type == "text")
            if verbose:
                print(f"[Paso {step}] Respuesta final del agente.")
            return texto_final

        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"[Paso {step}] Actuando: {block.name}({block.input})")
                fn = implementations.get(block.name)
                result = fn(**block.input) if fn else f"Herramienta desconocida: {block.name}"
                if verbose:
                    print(f"[Paso {step}] Observando resultado: {result}")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})

        messages.append({"role": "user", "content": tool_results})

    return "Se alcanzó el límite de pasos (max_steps) sin una respuesta final."


## 2. Paso 1 — Define tu caso de uso

Completa esto con tu equipo antes de programar:


In [ ]:
caso_de_uso = {
    "problema": "TODO: ¿qué problema real de la universidad resuelve tu agente?",
    "usuario": "TODO: ¿quién lo usaría? (estudiante, profesor, administrativo)",
    "exito": "TODO: ¿cómo sabes que el agente respondió bien?",
}
caso_de_uso


## 3. Paso 2 — Define las herramientas que tu agente necesitará

Sigue el mismo patrón: cada herramienta necesita un `name`, una `description` clara (de esto depende que el modelo la use bien) y una implementación real.


In [ ]:
# TODO: reemplaza estas herramientas de ejemplo por las de tu reto

def herramienta_1(parametro: str) -> str:
    """TODO: implementa la lógica real (consulta a una API, a una base de datos, etc.)"""
    return f"Resultado simulado para: {parametro}"

herramientas_taller = [
    {
        "name": "herramienta_1",
        "description": "TODO: describe cuándo debe usarse esta herramienta",
        "input_schema": {
            "type": "object",
            "properties": {"parametro": {"type": "string"}},
            "required": ["parametro"]
        }
    },
]

implementaciones_taller = {
    "herramienta_1": herramienta_1,
}


## 4. Paso 3 — Ejecuta el agente con tus herramientas

In [ ]:
respuesta = run_agent(
    "TODO: escribe aquí una pregunta real de prueba para tu agente",
    herramientas_taller,
    implementaciones_taller,
)
print(respuesta)


## 5. Paso 4 — Casos de prueba (mínimo 3, según el checklist de entrega)

In [ ]:
casos_de_prueba = [
    {"pregunta": "TODO caso 1", "resultado_esperado": "TODO", "resultado_obtenido": None},
    {"pregunta": "TODO caso 2", "resultado_esperado": "TODO", "resultado_obtenido": None},
    {"pregunta": "TODO caso 3", "resultado_esperado": "TODO", "resultado_obtenido": None},
]

for caso in casos_de_prueba:
    caso["resultado_obtenido"] = run_agent(caso["pregunta"], herramientas_taller, implementaciones_taller, verbose=False)

for caso in casos_de_prueba:
    print("Pregunta:", caso["pregunta"])
    print("Resultado:", caso["resultado_obtenido"])
    print("---")


## 6. Paso 5 — Reflexión final (entregable)

Responde brevemente, en base a lo visto en la charla:

1. **¿Qué límites o riesgos identificaron?** (loops infinitos, alucinación de herramientas, costos, datos sensibles...)
2. **¿Qué nivel de autonomía eligieron para el agente?** (asistido, human-in-the-loop, supervisado, autónomo) ¿Por qué?
3. **¿Qué agregarían si tuvieran una sesión más?** (ej: evaluación automática, observabilidad, exponerlo vía MCP)

*Estos tres puntos se profundizan en la Sesión 2 (Evaluación, Observabilidad) y en el Taller 6 (Seguridad y costos).*
